# IBD Registry: Post Extraction Data Cleaning, Cohort Construction, and Feature Engineering

This notebook documents the data extraction logic used for the Houston Methodist Hospital system's Epic electronic medical record database, along with the Python workflow for data cleaning, cohort construction, and feature engineering.

The workflow includes:

- Documentation of structured data extraction criteria
- Data preprocessing and cleaning
- Cohort construction
- Lab value preprocessing and categorization
- Demographic variable standardization
- Medication feature construction
- Clinical note and term extraction
- Construction of the final patient level analytic dataset

The original institutional SQL queries are not included because they depend on the local implementation of the Epic Caboodle and OnBase database schemas and operate on protected patient level data. Diagnosis codes, medication lists, and variable definitions are provided separately in Supplementary Table 1.

`PatientDurableKey` was used throughout the workflow to join extracted tables at the patient level. It was used only for table joins and was excluded before model training.

Patient level data are not included in the public repository. The Python cells expect input dataframes generated from the structured extraction steps documented below.

In [ ]:
import re
import numpy as np
import pandas as pd

In [ ]:
import os
import sys

data_path = "../../data"
os.makedirs(data_path, exist_ok=True)

### Extraction of IBD Diagnosis Records

IBD diagnosis records were extracted from the Epic Caboodle database using SQL. Crohn's disease records were defined by ICD-10 codes with K50.xx and ulcerative colitis by codes K51.xx

Each diagnosis was classified as:

* Crohn's disease when the ICD-10 code began with K50
* Ulcerative colitis when the ICD-10 code began with K51

The extracted diagnosis records included billing diagnoses, medical history diagnoses, External Claim Diagnoses, hospital problems, outgoing claim diagnoses, encounter diagnoses, problem list diagnoses, Problem List from External Source, Encounter Diagnosis from External Source, and Discharge Diagnosis from External Source.

The diagnosis record extraction included:

- `PatientDurableKey`: Patient ID
- `StartDate`: Diagnosis start date
- `UserEnteredDate`: User entered diagnosis date
- ICD-10 diagnosis code
- Derived CD or UC classification


### Patient Level IBD Diagnosis Summary

The diagnosis records were grouped by patient and aggregated using SQL.

The patient level dataset included:
- `PatientDurableKey` : Patient id
- `diagnosisStartDate`: Earliest diagnosis start date
- `diagnosisEnterDate`: Earliest user entered diagnosis date
- `CD`: Number of Crohn's disease diagnosis records with ICD-10 codes `K50.xx`
- `UC`: Number of ulcerative colitis diagnosis records with ICD-10 codes `K51.xx`
- `CDUCWeight`: Combined total number of Crohn's disease and ulcerative colitis diagnosis records with ICD-10 codes with `K50.xx` or `K51.xx`


The diagnosis count variables represent counts of diagnosis records.

### Patient demographics

Patient demographic information was extracted from the Epic Caboodle database and joined with the patient level IBD diagnosis summary using `PatientDurableKey`.

The demographic join added the following variables:

- `PrimaryMrn`
- `Name`
- `Gender`
- `BirthDate`
- `AgeInYears`
- `Ethnicity`
- `FirstRace`
- `PostalCode`
- `City`
- `StateOrProvince`
- `DeathDate`

After the join, the dataset contained the earliest diagnosis start date, earliest user entered diagnosis date, and Crohn's disease and ulcerative colitis diagnosis counts.

`PatientDurableKey`, `PrimaryMrn`, `Name`, `BirthDate`, `DeathDate`, and geographic variables were used only for cohort description and were excluded from the variables set before model training.


### Internal IBD Diagnosis Counts

ICD-10 diagnosis records were grouped by patient and aggregated using SQL.

A diagnosis record was included when:

- The ICD-10 code with `K50.xx` or `K51.xx`
- The diagnosis type did not contain the term `external`

For this analysis, records meeting the second criterion were categorized as internal diagnosis records. These records included billing diagnoses, medical history, hospital problems, outgoing claim diagnoses, encounter diagnoses, problem-list diagnoses, and labor-complication diagnoses.

Records with ICD-10 codes with `K50.xx` were classified as Crohn's disease, whereas records with `K51.xx` were classified as ulcerative colitis.

Variables created:

- `PatientDurableKey` : Patient id
- `CD_i`: Number of internal Crohn's disease diagnosis records for each patient
- `UC_i`: Number of internal ulcerative colitis diagnosis records for each patient
- `CDUCWeight_i`: Total number of internal Crohn's disease and ulcerative colitis diagnosis records for each patient


### External IBD Diagnosis Counts

ICD-10 diagnosis records were grouped by patient and aggregated using SQL.

A diagnosis record was included when:

- The ICD-10 code with `K50.xx` or `K51.xx`
- The diagnosis type with `external`, including external claim, problem-list, encounter, and discharge diagnosis records from external sources

Records with ICD-10 codes with `K50.xx` were classified as Crohn's disease, whereas records with `K51.xx` were classified as ulcerative colitis.

Variables:

- `PatientDurableKey` : Patient id
- `CD_e`: Number of external Crohn's disease diagnosis records.
- `UC_e`: Number of external ulcerative colitis diagnosis records.
- `CDUCWeight_e`: Total number of included external Crohn's disease and ulcerative colitis diagnosis.

### External Claim Diagnosis Count

ICD-10 diagnosis records were grouped by patient and counted using SQL.

A diagnosis record was included when:

- The ICD-10 code with `K50.xx` or `K51.xx`
- The diagnosis type was `External Claim Diagnosis`

Variables created:

- `PatientDurableKey`
- `externaldiagnosis_count`: Number of Crohn's disease and ulcerative colitis external claim diagnosis records for each patient


### IBD Specialist Visits from Billing Records

Billing transaction records were used to count the distinct dates on which each patient was seen by a provider from a specified list of IBD specialists.

created the following variables:

- `PatientDurableKey`
- `timesSeenbyIBDProv`: Total number of distinct service dates with the specified IBD specialist providers

### Houston Methodist Encounter Count

Encounter records were grouped by patient and aggregated using SQL. Encounters occurred within the Houston Methodist Hospital system were counted.

Variables:

- `PatientDurableKey` : Patient id
- `NumHMEncounters`: Total number of encounter records in the Houston Methodist health system for each patient

### Seen by a Dietitian

Patients with at least one billing transaction associated with the specified dietitian were considered to have seen a dietitian.

The following variable was created:

- `PatientDurableKey` : Patient id
- `seenbyDietitian`: `1` when at least one billing transaction was found and `0` otherwise

### Outpatient Visits with IBD Providers

Visit records were used to count outpatient visits with a specified list of IBD providers.

A visit was included when:

- The department specialty was Gastroenterology
- The primary visit provider was one of the specified IBD providers
- The appointment status was not Canceled, No Show, Left without seen, or Scheduled

Distinct appointment dates were counted for each patient.

Variables:

- `PatientDurableKey`
- `outpatientvisits_SeenbyIBDProv`: Number of distinct outpatient appointment dates with an included IBD provider
- `IBD_OP`: Binary Indicator equal to `1` when the patient had at least one outpatient visit and `0` otherwise

### Future Outpatient Visits with IBD Providers

Scheduled visits with a specified list of IBD specialist providers were extracted using SQL.

A visit was included when:

- The department specialty was Gastroenterology
- The appointment status was `Scheduled`
- The primary visit provider was one of the included IBD providers
- The visit had a future scheduled appointment date

Distinct future appointment dates were counted for each patient.

The following variable was created:

- `PatientDurableKey`
- `futureoutpatientvisits_SeenbyIBDProv`: Number of unique future outpatient appointment dates scheduled with an IBD specialist provider

### New Patient Future Visits with IBD Providers

Scheduled visits with a specified list of IBD specialist providers were extracted.

Patients were considered new to the IBD providers when they had no prior outpatient visits with those providers.

A future visit was included when:

- The department specialty was Gastroenterology
- The appointment status was `Scheduled`
- The primary visit provider was one of the included IBD specialist providers
- The visit had a future scheduled appointment date

The following variable was created:

- `PatientDurableKey`
- `newpatientFuturevisits_SeenbyIBDProv`: Binary indicator equal to `1` when a patient had at least one future appointment scheduled with an included IBD specialist provider and no prior outpatient visit with those providers; otherwise `0`

### Visits to the Gastroenterology Clinic

Visit records were used to count patient visits to the "GAST HMH SM 1201" gastroenterology department.

A visit was included when:

- The department name contained `GAST HMH SM 1201`
- The appointment status was not `Canceled`, `No Show`, `Left without seen`, or `Scheduled`
- A check-in date was available

The following variable was created:

- `PatientDurableKey`
- `SeenbyGastSM12provider`: Number of visits to the selected gastroenterology outpatient clinic for each patient

### IBD Medications Prescription Order History

IBD medications were extracted from medication order records.

The following medications were extracted:

Azathioprine, Balsalazide, Mercaptopurine, Mesalamine, Methotrexate, Olsalazine, Ozanimod, Sulfasalazine, Tofacitinib, Upadacitinib, Etrasimod, Natalizumab, Adalimumab, Certolizumab pegol, Golimumab, Infliximab, Mirikizumab, Risankizumab, Ustekinumab, Vedolizumab.

Created Variables:

- `PatientDurableKey`
- Standardized medication name
- Medication start date
- Medication end date


Multiple medication order periods for the same patient and medication were combined as semicolon separated date intervals in the format `YYYYMMDD-YYYYMMDD` or `YYYYMMDD-ONGOING`.

One intermediate medication history column was created for each medication. These columns were subsequently converted into binary medication indicators during Python preprocessing.

### IBD Meds Prescription Order Counts

Prescription order records were grouped by patient and standardized medication name for the following medications:

Adalimumab, Azathioprine, Balsalazide, Certolizumab pegol, Golimumab, Infliximab, Mercaptopurine, Mesalamine, Methotrexate, Mirikizumab, Natalizumab, Olsalazine, Ozanimod, Risankizumab, Sulfasalazine, Tofacitinib, Upadacitinib, Ustekinumab, Vedolizumab, and Etrasimod.

One count variable was created for each medication using the suffix `_medcount`.

Examples:

- `PatientDurableKey`
- `Adalimumab_medcount`: Number of Adalimumab prescription orders
- `Infliximab_medcount`: Number of Infliximab prescription orders
- `Olsalazine_medcount`: Number of Olsalazine prescription orders

The same definition was applied to the remaining medication count variables.

### Steroid Prescription Order History

Steroid prescription orders were extracted from medication order records.

The following steroid medications and administration routes were included:

- `Budesonide`: injection, oral, or rectal
- `Hydrocortisone`: buccal, feeding tube, g-tube, intramuscular, intravenous, j-tube, nasogastric, oral, or rectal
- `Methylprednisolone`: feeding tube, g-tube, injection, intravenous, j-tube, nasogastric, or oral
- `Prednisone`: feeding tube, g-tube, j-tube, nasogastric, or oral

The following information was included for each order:

- `PatientDurableKey`
- Medication name
- Medication start date
- Medication end date

Multiple prescription periods for the same patient and steroid medication were combined as semicolon-separated date intervals in the format `YYYYMMDD-YYYYMMDD`.

One intermediate medication-history column was created for each steroid medication:

- `PatientDurableKey` : Patient id
- `Budesonide`
- `Hydrocortisone`
- `Methylprednisolone`
- `Prednisone`

These columns were subsequently converted into binary medication indicators during Python preprocessing.

### Steroid Prescription Order Counts

Steroid prescription orders were counted for the following medications using the same medication-name and administration-route criteria applied to the steroid prescription-history variables:

- `Budesonide`
- `Hydrocortisone`
- `Methylprednisolone`
- `Prednisone`

Prescription-order records were grouped by patient and medication.

One count variable was created for each steroid medication using the suffix `_medcount`.


- `PatientDurableKey` : Patient id
-
Examples:
- `Budesonide_medcount`: Number of Budesonide prescription orders
- `Hydrocortisone_medcount`: Number of Hydrocortisone prescription orders
- `Methylprednisolone_medcount`: Number of Methylprednisolone prescription orders
- `Prednisone_medcount`: Number of Prednisone prescription orders

### BMI

BMI values were extracted from flowsheet records using BMI related flowsheet names, including `BMI`, `BMI (Calculated)`, `BMI (RD Calculated)`, and `BMI:`.

Only numeric BMI values between 12 and 50 were included.

The following information was extracted for each BMI record:

- `PatientDurableKey` : Patient id
- BMI measurement date and time
- Numeric BMI value
- Flowsheet display name

### BMI Preprocessing

The `BMI` dataframe represents the output of the structured BMI extraction step described above.
The dataframe is expected to contain the following columns:

- `PatientDurableKey`
- `TakenInstant`
- `NumericValue`
- `DisplayName`

In [ ]:
# Expected input: BMI table generated from the structured BMI extraction step

import pandas as pd
BMI_df = BMI.copy()
BMI_df = BMI_df[BMI_df['TakenInstant'] != '-1']

BMI_df['TakenInstant'] = pd.to_datetime(BMI_df['TakenInstant'])

median_bmi = BMI_df.groupby('PatientDurableKey')['NumericValue'].median().round(2).reset_index(name='median_bmi')
median_bmi

### Labs

Labs were extracted from structured lab records.

The following labs were included:

`CRP`, `Fecal lactoferrin`, `Fecal calprotectin`, and `Sedimentation rate`.

Variables created:

- PatientDurableKey
- Lab name
- Lab value
- Collection date

### Labs Preprocessing

The `Labs` dataframe represents the output of the structured laboratory extraction step described above.

The dataframe is expected to contain the following columns:

- `PatientDurableKey`
- `Name`
- `Value`
- `CollectionDate`

In [ ]:

# Labs Preprocessing
# Lab  values were converted into numeric form. Leading symbols such as `<` and `>` were removed. Negative or not-detected fecal lactoferrin results were assigned `0`, and positive or detected results were assigned `1`.
# `Sedimentation rate` was standardized as `ESR`.
# For each patient and lab, the maximum available processed value was included. The results were then converted to a patient level table with one column per laboratory component.

# Expected input: Labs generated from the structured laboratory extraction step
Labsdf = Labs.copy()


def process_lab_value(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Remove leading symbols
    if value.startswith(("<", ">")):
        value = value[1:].strip()

    value_lower = value.lower()

    negative_values = {"negative","not detected","no detected","no fecal lactoferrin detected",}
    positive_values = {"positive","fecal lactoferrin detected",}

    if value_lower in negative_values:
        return 0.0

    if value_lower in positive_values:
        return 1.0

    return pd.to_numeric(value, errors="coerce")

Labsdf["Name"] = Labsdf["Name"].replace({"Sedimentation rate": "ESR"})
Labsdf["processed_lab_value"] = Labsdf["Value"].apply(process_lab_value)


valid_labs = Labsdf.dropna(subset=["processed_lab_value"]).copy()

idx = (valid_labs.groupby(["PatientDurableKey", "Name"])["processed_lab_value"].idxmax())
max_lab_values = valid_labs.loc[idx,["PatientDurableKey","Name","processed_lab_value","CollectionDate",],].copy()

labsdf_pivot = (
    max_lab_values.pivot(
        index="PatientDurableKey",
        columns="Name",
        values="processed_lab_value",
    ).reset_index())

labsdf_pivot.columns.name = None
labsdf_pivot

### Extraintestinal Manifestation and Related Diagnosis Counts

Selected extraintestinal manifestations and related GI conditions were extracted from ICD-10 diagnosis records.
The following diagnosis groups and ICD-10 codes were included:

| Diagnosis group                             | ICD-10 codes                                            |
|---------------------------------------------|---------------------------------------------------------|
| Peripheral arthropathies                    | `M19.90`, `M13.8`, `M07.60`, `M13.0`, `M06.4`, `M46.90` |
| Axial arthropathies                         | `M45.00`, `M46.1`                                       |
| Erythema nodosum                            | `L52.xx`                                                |
| Pyoderma gangrenosum                        | `L88`                                                   |
| Sweet syndrome                              | `L98.2`                                                 |
| Aphthous stomatitis                         | `K12.0`                                                 |
| Primary sclerosing cholangitis              | `K83.01`                                                |
| Episcleritis                                | `H15.1`                                                 |
| Anterior uveitis                            | `H20.01`, `H20.02`, `H20.04`                            |
| Iritis                                      | `H20.029`                                               |
| Scleritis                                   | `H15.0`                                                 |
| Psoriasis                                   | `L40.0`, `L40.1`, `L40.2`, `L40.3`, `L40.4`             |
| Psoriatic arthritis                         | `L40.5`                                                 |
| Pressure ulcer                              | `L89`                                                   |
| NP chronic ulcer of the lower limb          | `L97`                                                   |
| NP chronic ulcer of the skin                | `L98.4`                                                 |
| Vascular disorders of the intestine         | `K55`                                                   |
| Diverticular disease of the intestine       | `K57`                                                   |
| Irritable bowel syndrome                    | `K58`                                                   |
| Clostridioides difficile enterocolitis      | `A04.7`                                                 |
| specified bacterial intestinal infections   | `A04.8`                                                 |
| Bacterial intestinal infection, unspecified | `A04.9`                                                 |
| bacterial intestinal infections             | `A04.0` - `A06.0`                                       |
| Infectious gastroenteritis and colitis      | `A09`                                                   |
| Colon cancer                                | `C18`                                                   |
| history of colon cancer                     | `Z85.03`, `Z85.04`                                      |

One count variable was created for each diagnosis group using the suffix `_seencount`.
- `PatientDurableKey` : Patient id
-
Examples:

- `Peripheralarthropathies_seencount`: Number of peripheral arthropathy diagnosis records
- `Primarysclerosischolangitis_seencount`: Number of primary sclerosing cholangitis diagnosis records
- `EnterocolitisClostridiumDifficile_seencount`: Number of *Clostridioides difficile* enterocolitis diagnosis records
- `coloncancer_seencount`: Number of colon cancer diagnosis records

### Endoscopy Procedures and Endoscopy records Count

Endoscopy procedures were extracted from structured procedure records using procedure names.

The following procedures were included:

Colonoscopy, endoscopy, esophagogastroduodenoscopy, proctosigmoidoscopy, sigmoidoscopy, pouchoscopy, enteroscopy, ileoscopy, anoscopy, esophagoscopy, gastroscopy, and jejunoscopy.

OP notes associated with the procedure encounters were selected using the clinical note type `Op Note`.

- `Numberofendoscopyprocedures`: Number of endoscopy procedure records for each patient
- `endoscopyrecords_count`: Number of distinct Op notes associated with procedure encounters

The final dataset included one row per patient while the procedure and op notes counts represented all records for that patient.

### Endoscopy OP Notes Extraction

Endoscopy related procedures were extracted from structured procedure records using procedure names.

The following procedures were included:

Colonoscopy, endoscopy, esophagogastroduodenoscopy, proctosigmoidoscopy, sigmoidoscopy, pouchoscopy, enteroscopy, ileoscopy, anoscopy, esophagoscopy, gastroscopy, and jejunoscopy.

Op Notes associated with procedure encounters were extracted from structured clinical records using the clinical note type `Op Note`.

Variables:

- `PatientDurableKey`
- procedure name
- Procedure start date
- Procedure end date
- Note type
- Op note text
- Note creation date and time


### Endoscopy Op Note Keyword Extraction

The extracted endoscopy op notes were preprocessed in Python for keyword extraction.

Regular-expression pattern matching was used to find the following endoscopy-related findings:

- Aphthous ulcer, Anastomosis ,Pouchitis ,Ulcer ,Erythema ,Loss of vascularity ,Friable mucosa ,Mayo, Contact bleeding, Rutgeerts, Ileo-colonic, SES-CD

Keyword occurrences were counted separately within each Op note and then summed across all OP notes for each patient.

All extracted records were associated with patients using `PatientDurableKey`. One patient-level count variable was created for each keyword using the prefix `Endoscopy_`.

Examples:

- `Endoscopy_Aphthous ulcer`
- `Endoscopy_ulcerated`
- `Endoscopy_erythema`
- `Endoscopy_SES-CD`

The same procedure was applied to the remaining keywords.

In [ ]:

# Expected input: endoscopy_clinicalnotes generated from the endoscopy OP Note extraction step

df = endoscopy_clinicalnotes.copy()
df = df.dropna(subset=["Text"]).copy()
df['CreationInstant'] = pd.to_datetime(df['CreationInstant'])

# Remove duplicate notes
# ClinicalNoteKey is preferred when available
if "ClinicalNoteKey" in df.columns:
    df = df.drop_duplicates(subset=["ClinicalNoteKey"])
else:
    df = df.drop_duplicates(
        subset=["PatientDurableKey", "CreationInstant", "Text"]
    )

# Regex patterns for keyword extraction
words_to_count = {
    'Aphthous ulcer': r'\baphthous ulcer(?:s|ations)?\b',
    'Anastomosis': r'\banastomosis\b',
    'Pouchitis': r'\bpouchitis\b',
    'ulcerated': None,
    'erythema': r'\berythema(?:tous)?\b',
    'Loss of vascularity': r'\bloss of vascularity\b',
    'Friable mucosa': r'\bfriable mucosa\b',
    'mayo': r'\bmayo\b',
    'contact bleeding': r'\bcontact bleeding\b',
    'rutgeerts': r'\brutgeerts\b',
    'Ileo-Colonic': r'\bileo[- ]colonic\b',
    'SES-CD': r'\bses[- ]cd\b'
}


# ulcer counter
def count_ulcers(text):
    pattern = r'\bulcer(?:ation|ated|ations|s)?\b'
    matches = re.finditer(pattern, text, flags=re.IGNORECASE)

    filtered_matches = [
        match.group(0)
        for match in matches
        if not re.search(r'\b(?:aphthous|no)\b\s*$', text[:match.start()], flags=re.IGNORECASE)
    ]

    return len(filtered_matches)


#  word counting
def count_words(text, patterns):
    text = str(text).lower()
    counts = {}
    for key, pattern in patterns.items():
        if key == 'ulcerated':
            counts[key] = count_ulcers(text)
        else:
            counts[key] = len(re.findall(pattern, text))
    return counts

def aggregate_counts(texts):
    total_counts = {key: 0 for key in words_to_count}
    for text in texts:
        counts = count_words(text, words_to_count)
        for key in counts:
            total_counts[key] += counts[key]
    return total_counts


results = (df.groupby('PatientDurableKey').agg({'Text': lambda x: aggregate_counts(list(x))}).reset_index())

results = results.rename(columns={'Text': 'Counts'})
expanded_df = results['Counts'].apply(pd.Series)
expanded_df = expanded_df.add_prefix('Endoscopy_')
endoscopywords_df = pd.concat([results[['PatientDurableKey']], expanded_df], axis=1)
endoscopywords_df


### Colon and Rectal Surgery Clinical Notes

Clinical notes were extracted from structured clinical records.

Colon and rectal surgery notes were selected based on the following criteria:

- The clinical service was `Colon & Rectal Surgery`

Created following variables:

- `PatientDurableKey`
- Clinical-note id
- Note type
- Clinical service
- Note text
- Service date and time
- Note creation date and time

### Colon and Rectal Surgery Note Keyword Filtering

Colon and rectal surgery clinical notes were searched for the following IBD related terms:

`ulcerative`, `ulcerative colitis`, `IBD`, `inflammatory bowel disease`, `crohn`, `crohns`, and `crohn's`.

Notes containing at least one matching term were included.

Notes were grouped by patient, and the following variable was created:

- `PatientDurableKey` : Patient id
- `colon&rectal_seencount`: Number of clinical note records from the Colon & Rectal Surgery service that contained at least one IBD keyword.

In [ ]:
import re

keywords = [
    'ulcerative',
    'ulcerative colitis',
    'IBD',
    'inflammatory bowel disease',
    'crohn',
    'crohns',
    "crohn's"
]

# Expected input: colonRectal dataframe generated from the Colon & Rectal Surgery clinical note extraction step

df = colonRectal.copy()
escaped_keywords = [re.escape(word) for word in keywords]


pattern = r'\b(?:' + '|'.join(escaped_keywords) + r')\b'
regex = re.compile(pattern, flags=re.IGNORECASE)

filtered_df = df[df['Text'].str.contains(regex, na=False)]

note_counts = filtered_df.groupby('PatientDurableKey').size().reset_index(name='colon&rectal_seencount')
colonrectal_notecounts = note_counts.copy()
colonrectal_notecounts


### Gastroenterology Progress and Consult Notes

Clinical notes were extracted from structured clinical records.

GI records were selected based on the following criteria:

- The note type was `Progress Notes` or `Consults`
- The clinical service was `Gastroenterology`

Following variables were created:

- PatientDurableKey
- Note type
- Clinical service
- Note text
- Service date and time
- Note creation date and time

### Gastroenterology Note Counts and Keyword Extraction

Gastroenterology progress and consult notes were processed in Python.

The total number of gastroenterology note records was calculated for each patient.

The notes were searched for the following prespecified terms:

`Ileitis`, `Abscess`, `Fistula`, `Stricture`, `Proctocolectomy`, `Colectomy`, `Ileostomy`, `Colitis`, `Wall Thickening`, `Ischemia`, `Ischemic`, `Ischaemia`, `Ischaemic`, `Ileocecal resection`, `Hemicolectomy`, `Diverting ileostomy`, `Proctosigmoiditis`, `Total colectomy`, `Total proctocolectomy`, `Ileorectal`, `Fibrostenotic`, `J-pouch`, `Pouchitis`, `Ileocolonic Anastomosis`, `Ileal Pouch Anal Anastomosis`, and `Ileoanal`.

Keyword occurrences were counted across all gastroenterology notes for each patient.

Variables:

- `PatientDurableKey` : Patient id
- `Gastroenterology_recordscount`: Number of gastroenterology clinical-note records
- `Gastroenterology_<keyword>`: Total number of occurrences of the corresponding keyword across all gastroenterology notes

Examples include:

- `Gastroenterology_Ileitis`
- `Gastroenterology_Fistula`
- `Gastroenterology_Stricture`
- `Gastroenterology_Colectomy`
- `Gastroenterology_Pouchitis`

In [ ]:

# Expected input: GINotes generated from the Gastroenterology Progress and Consult Notes extraction step
df = GINotes.copy()
df = df.dropna(subset=["Text"]).copy()
df['CreationInstant'] = pd.to_datetime(df['CreationInstant'])

output = df.groupby('PatientDurableKey')['Text'].count().reset_index()
output = output.rename(columns={'Text': 'Gastroenterology_recordscount'})

# Keywords and regex patterns
words_to_count = {
    'Ileitis': r'\bileitis\b',
    'Abscess': r'\babscess\b',
    'Fistula': r'\bfistul(?:a|izing)?\b',
    'Stricture': r'\bstricture\b',
    'Proctocolectomy': r'\bproctocolectomy\b',
    'Colectomy': r'\bcolectomy\b',
    'Ileostomy': r'\bileostomy\b',
    'Colitis': r'\bcolitis\b',
    'Wall Thickening': r'\bwall thickening\b',
    'Ischemia': r'\bischemia\b',
    'Ischemic': r'\bischemic\b',
    'Ischaemia': r'\bischaemia\b',
    'Ischaemic': r'\bischaemic\b',
    'Ileocecal resection': r'\bileocecal resection\b',
    'Hemicolectomy': r'\bhemicolectomy\b',
    'Diverting ileostomy': r'\bdiverting ileostomy\b',
    'Proctosigmoiditis': r'\bproctosigmoiditis\b',
    'Total colectomy': r'\btotal colectomy\b',
    'Total proctocolectomy': r'\btotal proctocolectomy\b',
    'Ileorectal': r'\bileorectal\b',
    'Fibrostenotic': r'\bfibrostenotic\b',
    'J-pouch': r'\bj[- ]pouch\b',
    'Pouchitis': r'\bpouchitis\b',
    'Ileocolonic Anastomosis': r'\bileocolonic anastomosis\b',
    'Ileal Pouch Anal Anastomosis': r'\bileal pouch anal anastomosis\b',
    'Ileoanal': r'\bileoanal\b'}


def count_words(text, patterns):
    text = str(text).lower()
    counts = {}
    for key, pattern in patterns.items():
        counts[key] = len(re.findall(pattern, text))
    return counts


results = (df.groupby('PatientDurableKey').agg({'Text': lambda x: count_words(' '.join(x), words_to_count)}).reset_index())

expanded_df = results['Text'].apply(pd.Series)
expanded_df = expanded_df.add_prefix('Gastroenterology_')
GI_df = pd.concat([results[['PatientDurableKey']], expanded_df], axis=1)

GI_df = GI_df.merge(output, on='PatientDurableKey', how='left')

cols = ['PatientDurableKey', 'Gastroenterology_recordscount'] + \
       [c for c in GI_df.columns if c not in ['PatientDurableKey', 'Gastroenterology_recordscount']]

GI_df = GI_df[cols]
GI_df

### Imaging Reports

Imaging reports were extracted from structured imaging records.

Imaging records were included when they met the following criteria:

- The order type was `Imaging`
- The imaging narrative was not empty
- The procedure short name matched one of the prespecified procedure names listed in Supplementary Table 1

Variables created:

- PatientDurableKey
- Imaging record ID
- Imaging order type
- Procedure short name
- Imaging narrative
- Imaging text creation date and time

### Imaging Report Counts and Keyword Extraction

Imaging report narratives were processed in Python.

The total number of imaging report narratives was calculated for each patient.

The reports were searched for the following prespecified terms:

`Ileitis`, `Abscess`, `Fistula`, `Stricture`, `Proctocolectomy`, `Colectomy`, `Ileostomy`, `Colitis`, `Wall Thickening`, `Ischemia`, `Ischemic`, `Ischaemia`, `Ischaemic`, `Ileocecal resection`, `Hemicolectomy`, `Diverting ileostomy`, `Proctosigmoiditis`, `Total colectomy`, `Total proctocolectomy`, `Ileorectal`, `Fibrostenotic`, `J-pouch`, `Pouchitis`, `Ileocolonic Anastomosis`, `Ileal Pouch Anal Anastomosis`, and `Ileoanal`.

Keyword occurrences were counted across all imaging narratives for each patient.

Variables:

- `PatientDurableKey` : Patient id
- `Imaging_recordscount`: Number of imaging reports
- `Imaging_<keyword>`: Total number of occurrences of the corresponding keyword across all imaging narratives.

In [ ]:
import re

# Expected input: imagingNotes dataframe generated from the Imaging Reports extraction step

df = imagingNotes.copy()
df = df.dropna(subset=["narrative"]).copy()
df['_CreationInstant'] = pd.to_datetime(df['_CreationInstant'])

note_counts = (df.groupby('PatientDurableKey')['narrative'].count().reset_index().rename(
    columns={'narrative': 'Imaging_recordscount'}))

# Regex patterns
words_to_count = {
    'Ileitis': r'\bileitis\b',
    'Abscess': r'\babscess\b',
    'Fistula': r'\bfistul(?:a|izing)?\b',
    'Stricture': r'\bstricture\b',
    'Proctocolectomy': r'\bproctocolectomy\b',
    'Colectomy': r'\bcolectomy\b',
    'Ileostomy': r'\bileostomy\b',
    'Colitis': r'\bcolitis\b',
    'Wall Thickening': r'\bwall thickening\b',
    'Ischemia': r'\bischemia\b',
    'Ischemic': r'\bischemic\b',
    'Ischaemia': r'\bischaemia\b',
    'Ischaemic': r'\bischaemic\b',
    'Ileocecal resection': r'\bileocecal resection\b',
    'Hemicolectomy': r'\bhemicolectomy\b',
    'Diverting ileostomy': r'\bdiverting ileostomy\b',
    'Proctosigmoiditis': r'\bproctosigmoiditis\b',
    'Total colectomy': r'\btotal colectomy\b',
    'Total proctocolectomy': r'\btotal proctocolectomy\b',
    'Ileorectal': r'\bileorectal\b',
    'Fibrostenotic': r'\bfibrostenotic\b',
    'J-pouch': r'\bj[- ]pouch\b',
    'Pouchitis': r'\bpouchitis\b',
    'Ileocolonic Anastomosis': r'\bileocolonic anastomosis\b',
    'Ileal Pouch Anal Anastomosis': r'\bileal pouch anal anastomosis\b',
    'Ileoanal': r'\bileoanal\b'
}

def count_intestinal(text):
    pattern = r'\bintestinal(?:\s*(?:us|ultrasound))?\b'
    matches = re.finditer(pattern, text, flags=re.IGNORECASE)
    filtered_matches = [match.group(0) for match in matches]
    return len(filtered_matches)

def count_words(text, patterns):
    text = str(text).lower()
    counts = {}
    for key, pattern in patterns.items():
        if key == 'Intestinal US':
            counts[key] = count_intestinal(text)
        else:
            counts[key] = len(re.findall(pattern, text))
    return counts


results = (df.groupby('PatientDurableKey').agg({'narrative': lambda x: count_words(' '.join(x), words_to_count)}).reset_index())

expanded_df = results['narrative'].apply(pd.Series)
expanded_df = expanded_df.add_prefix('Imaging_')
imagingnotes_df = pd.concat([results[['PatientDurableKey']], expanded_df], axis=1)

imagingnotes_df = imagingnotes_df.merge(note_counts, on='PatientDurableKey', how='left')

cols = ['PatientDurableKey', 'Imaging_recordscount'] + \
       [c for c in imagingnotes_df.columns if c not in ['PatientDurableKey', 'Imaging_recordscount']]

imagingnotes_df = imagingnotes_df[cols]
imagingnotes_df

### Pathology Report Counts and Keyword Extraction

Pathology reports were extracted from the institutional OnBase database and processed in Python.

The total number of pathology reports was calculated for each patient.

The reports were searched for the following prespecified terms:

`Chronic`, `Ileitis`, `Pouchitis`, `Colitis`, `Granuloma`, `Granulomatous`, `Crypt distortion`, and `Crypt Abscess`.

Keyword occurrences were counted across all pathology reports for each patient.

Variables:

- PatientDurableKey
- `pathology_recordscount`: Number of pathology report records for each patient
- `pathology_<keyword>`: Total number of occurrences of the keyword across all pathology reports for each patient

In [ ]:

import re

# Expected input: pathology dataframe extracted from the institutional OnBase database
df = pathologyLabs.copy()

df = df.dropna(subset=["reports"]).copy()
df["reports"] = df["reports"].astype(str)


note_counts = (
    df.groupby("PatientDurableKey")
    .size()
    .reset_index(name="pathology_recordscount")
)

# Regex patterns
words_to_count = {
    'Chronic': r'\bchronic\b',
    'Ileitis': r'\bileitis\b',
    'Pouchitis': r'\bpouchitis\b',
    'Colitis': r'\bcolitis\b',
    'Granuloma': r'\bgranuloma\b',
    'Granulomatous': r'\bgranulomatous\b',
    'Crypt distortion': r'\bcrypt distortion\b',
    'Crypt Abscess': r'\bcrypt abscess\b'}


def count_words(text, patterns):
    text = str(text).lower()
    counts = {}
    for key, pattern in patterns.items():
        counts[key] = len(re.findall(pattern, text))
    return counts


results = (
    df.groupby('PatientDurableKey')
    .agg({'reports': lambda x: count_words(' '.join(x), words_to_count)})
    .reset_index())

expanded_df = results['reports'].apply(pd.Series)
expanded_df = expanded_df.add_prefix('pathology_')
pathologyrecordsdf = pd.concat([results[['PatientDurableKey']], expanded_df], axis=1)

pathologyrecordsdf = pathologyrecordsdf.merge(note_counts, on='PatientDurableKey', how='left')

cols = ['PatientDurableKey', 'pathology_recordscount'] + \
       [c for c in pathologyrecordsdf.columns if c not in ['PatientDurableKey', 'pathology_recordscount']]

pathologyrecordsdf = pathologyrecordsdf[cols]
pathologyrecordsdf

### Preprocessing

### Integration of Patient Level Feature Tables

The demographic and IBD diagnosis dataset was used as the base patient cohort. All feature tables used in this integration were generated from the extraction and preprocessing steps described in the preceding sections of this notebook.

Internal and external diagnosis counts, visits, medication features, BMI and labs, related diagnosis counts, procedure variables, and clinical text features were merged using `PatientDurableKey`.

All feature tables were expected to contain one row per patient before merging.

The resulting dataframe, `allvariables_df`, contained one row per patient and all extracted or derived patient level variables.

In [ ]:

# Create the base patient cohort

# Source: Demographic extraction and IBD diagnosis summary
# Contains: patient demographics, diagnosis dates, CD and UC diagnosis counts
allvariables_df = IBDRegistry_CDUC_demographics.copy()

# Source: Internal IBD Diagnosis Counts extraction - Contains: internal CD and UC diagnosis counts
allvariables_df = allvariables_df.merge(IBDRegistry_CDUCcount_internal,on="PatientDurableKey", how="left")

# Source: External IBD Diagnosis Counts extraction - Contains: External CD and UC diagnosis counts
allvariables_df = allvariables_df.merge(IBDRegistry_CDUCcount_external,on="PatientDurableKey",how="left")

# Prepare selected patient-level feature tables
# Source: Services Associated with IBD Specialist Providers - Contains: timesSeenbyIBDProv
billingproviders_df = billingproviders_timesseenbyIBD[["PatientDurableKey", "timesSeenbyIBDProv"]].copy()
# Source:  New Patient Future Visits with IBD Providers extraction - Contains: newpatientFuturevisits_SeenbyIBDProv
newpatient_future_df = newpatientFuturevisits_SeenbyIBDProv[["PatientDurableKey", "newpatientFuturevisits_SeenbyIBDProv"]].copy()
# Source:  Endoscopy Procedures and Endoscopy records Count extraction - Contains: Numberofendoscopyprocedures, endoscopyrecords_count
endoscopy_counts_df = endoscopy_proceduresRecordscount[["PatientDurableKey","Numberofendoscopyprocedures","endoscopyrecords_count"]].copy()

# Source: External Claim Diagnosis Count extraction - Contains: externaldiagnosis_count
allvariables_df = allvariables_df.merge(externaldiagnosiscount,on="PatientDurableKey",how="left")

# Source: Services Associated with IBD Specialist Providers - Contains: timesSeenbyIBDProv
allvariables_df = allvariables_df.merge(billingproviders_df,on="PatientDurableKey",how="left")

# Source: Houston Methodist Encounter Count extraction - Contains: NumHMEncounters
allvariables_df = allvariables_df.merge(NumHMEncounters,on="PatientDurableKey",how="left")
# Source: Seen by a Dietitian extraction - Contains: seenbyDietitian
allvariables_df = allvariables_df.merge(dietitianrecords,on="PatientDurableKey",how="left")
# Source: Gastroenterology Visits with IBD Providers extraction - Contains: outpatientvisits_SeenbyIBDProv, IBD_OP
allvariables_df = allvariables_df.merge(outpatientvisits_seenbyIBD,on="PatientDurableKey",how="left")

# Source: Future Outpatient Visits with IBD Providers extraction - Contains: futureoutpatientvisits_SeenbyIBDProv
allvariables_df = allvariables_df.merge(future_outpatientvisits,on="PatientDurableKey", how="left")

# Source:  New Patient Future Visits with IBD Providers extraction - Contains: newpatientFuturevisits_SeenbyIBDProv
allvariables_df = allvariables_df.merge(newpatient_future_df,on="PatientDurableKey",how="left")
# Source:  Visits to the Gastroenterology Clinic extraction - Contains: SeenbyGastSM12provider
allvariables_df = allvariables_df.merge(SeenbyGastSM12provider,on="PatientDurableKey",how="left")

# Merge medication features
# Source: IBD Meds Prescription Order History extraction - Contains: patient level IBD medication history variables
allvariables_df = allvariables_df.merge(IBDMeds,on="PatientDurableKey",how="left")
# Source:  Steroid Prescription Order History extraction - Contains: patient level steroid prescription history variables
allvariables_df = allvariables_df.merge(budesonide_steroid,on="PatientDurableKey",how="left")
allvariables_df = allvariables_df.merge(hydrocortisone_steroid,on="PatientDurableKey",how="left")
allvariables_df = allvariables_df.merge(methylprednisolone_steroid,on="PatientDurableKey",how="left")
allvariables_df = allvariables_df.merge(prednisone_steroid,on="PatientDurableKey",how="left")

# Merge medication-order counts
# Source: IBD Meds Prescription Order Counts extraction - Contains: patient level IBD medication order counts
allvariables_df = allvariables_df.merge(IBDMeds_count,on="PatientDurableKey",how="left")
# Source:  Steroid Prescription Order Counts extraction - Contains: patient level steroid prescription order counts
allvariables_df = allvariables_df.merge(BudesonideMedcount,on="PatientDurableKey",how="left")
allvariables_df = allvariables_df.merge(HydrocortisonemedCount,on="PatientDurableKey",how="left")
allvariables_df = allvariables_df.merge(MethylprednisolonemedCount, on="PatientDurableKey",how="left")
allvariables_df = allvariables_df.merge(Prednisonemedcount,on="PatientDurableKey",how="left")

# Merge BMI, Labs, and other diagnosis features
# Source:  BMI Preprocessing step - Contains: median_bmi
allvariables_df = allvariables_df.merge(median_bmi,on="PatientDurableKey",how="left")
# Source:  Laboratory Preprocessing step - Contains:  maximum CRP, ESR, fecal calprotectin, and fecal lactoferrin values
allvariables_df = allvariables_df.merge(labsdf_pivot,on="PatientDurableKey",how="left")
# Source: Extraintestinal Manifestation and Related Diagnosis Counts extraction - Contains: patient level counts of related and extraintestinal diagnosis records
allvariables_df = allvariables_df.merge(new_diagnosiscodesdata,on="PatientDurableKey",how="left")

# Merge procedure and clinical-text features

# Source:  Endoscopy Procedures and Endoscopy records Count extraction - Contains: Numberofendoscopyprocedures, endoscopyrecords_count
allvariables_df = allvariables_df.merge(endoscopy_counts_df,on="PatientDurableKey",how="left")
# Source:  Endoscopy Op Note Keyword preprocessing step - Contains: patient level endoscopy op note keyword counts
allvariables_df = allvariables_df.merge(endoscopywords_df,on="PatientDurableKey",how="left")
# Source:  Colon and Rectal Surgery Note Keyword Filtering preprocessing step  - Contains: colon&rectal_seencount
allvariables_df = allvariables_df.merge(colonrectal_notecounts,on="PatientDurableKey",how="left")
# Source:  Gastroenterology Note Counts and Keyword Extraction preprocessing step - Contains: patient level gastroenterology note count and keyword counts
allvariables_df = allvariables_df.merge(GI_df,on="PatientDurableKey",how="left")
# Source:  Imaging Report Counts and Keyword Extraction preprocessing step  - Contains: patient level imaging records & keywords count
allvariables_df = allvariables_df.merge(imagingnotes_df,on="PatientDurableKey",how="left")
# Source:  Pathology Report Counts and Keyword Extraction preprocessing step  - Contains: patient level pathology records & keywords count
allvariables_df = allvariables_df.merge(pathologyrecordsdf,on="PatientDurableKey",how="left")


allvariables_df

### Medication Exposure Indicators and Counts

Medication history variables generated in the preceding extraction steps contained date range information for each medication. Each variable was converted into a binary exposure indicator:

- `1` indicated that medication history was available for the patient.
- `0` indicated that no medication history record.

The binary medication indicators were then summarized into three patient-level variables:

- `steroids_med_weight`: number of distinct steroid medications recorded
- `nonsteroids_med_weight`: number of distinct nonsteroid IBD medications recorded
- `total_med_weight`: total number of distinct medication exposures recorded

In [ ]:

# Use the integrated patientlevel dataframe created above
df = allvariables_df.copy()

date_columns = [
    'Adalimumab', 'Certolizumabpegol', 'Etrasimod', 'Golimumab', 'Infliximab', 'Mirikizumab',
    'Natalizumab', 'Olsalazine', 'Ozanimod', 'Risankizumab', 'Tofacitinib', 'Upadacitinib',
    'Ustekinumab', 'Vedolizumab', 'Azathioprine', 'Balsalazide', 'Mercaptopurine', 'Mesalamine',
    'Methotrexate', 'Sulfasalazine', 'Budesonide', 'Hydrocortisone', 'Methylprednisolone', 'Prednisone']

# df[date_columns] = df[date_columns].replace(r"^\s*$",pd.NA,regex=True)
df[date_columns] = df[date_columns].notna().astype(int)

# Meds weight count

medication_columns = ['Adalimumab', 'Certolizumabpegol', 'Etrasimod', 'Golimumab', 'Infliximab', 'Mirikizumab',
                      'Natalizumab', 'Olsalazine', 'Ozanimod', 'Risankizumab', 'Tofacitinib', 'Upadacitinib',
                      'Ustekinumab', 'Vedolizumab', 'Azathioprine', 'Balsalazide', 'Budesonide', 'Hydrocortisone',
                      'Mercaptopurine', 'Mesalamine', 'Methotrexate', 'Methylprednisolone', 'Prednisone',
                      'Sulfasalazine']

selected_medications = ['Budesonide', 'Hydrocortisone', 'Methylprednisolone', 'Prednisone']


# number of distinct meds for each patient
def count_medications(row):
    steroids_count = sum(row[col] == 1 for col in selected_medications)
    nonsteroids_count = sum(
        row[col] == 1 for col in medication_columns
        if col not in selected_medications)

    total_count = steroids_count + nonsteroids_count
    return pd.Series([steroids_count, nonsteroids_count, total_count])


df[['steroids_med_weight', 'nonsteroids_med_weight', 'total_med_weight']] = df.apply(count_medications, axis=1)

med_datesweight = df.copy()
med_datesweight

### Medication Class Prescription Counts and Exposure Indicators

Medication order count variables generated in the preceding extraction steps were grouped into prespecified therapeutic classes.

For each medication class, two patient level variables were created:

- `<class>_prescriptioncount`: total number of prescription orders across medications within the class
- `<class>`: binary indicator equal to `1` when at least one prescription order was found and `0` otherwise

The medication classes included aminosalicylates, immunomodulators, anti-TNF therapies, anti-interleukin therapies, anti-integrin therapies, JAKI inhibitors, sphingosine-1-phosphate receptor modulators, and steroid.

The `Steroid` medication class variable was based only on budesonide prescription orders. Hydrocortisone, methylprednisolone, and prednisone were included as separate medication variables but were not included in this class-level indicator.

In [ ]:
df = med_datesweight.copy()

medication_columns = [
    'Adalimumab_medcount', 'Certolizumabpegol_medcount', 'Etrasimod_medcount', 'Golimumab_medcount',
    'Infliximab_medcount', 'Mirikizumab_medcount', 'Natalizumab_medcount', 'Olsalazine_medcount',
    'Ozanimod_medcount', 'Risankizumab_medcount', 'Tofacitinib_medcount', 'Upadacitinib_medcount',
    'Ustekinumab_medcount', 'Vedolizumab_medcount', 'Azathioprine_medcount', 'Balsalazide_medcount',
    'Budesonide_medcount', 'Hydrocortisone_medcount', 'Mercaptopurine_medcount', 'Mesalamine_medcount',
    'Methotrexate_medcount', 'Methylprednisolone_medcount', 'Prednisone_medcount', 'Sulfasalazine_medcount'
]

df[medication_columns] = df[medication_columns].apply(pd.to_numeric, errors='coerce')

medication_groups = {
    "Aminosalicylates": [
        'Sulfasalazine_medcount', 'Mesalamine_medcount',
        'Balsalazide_medcount', 'Olsalazine_medcount'
    ],
    "Immunomodulators": [
        'Azathioprine_medcount', 'Mercaptopurine_medcount', 'Methotrexate_medcount'
    ],
    "Anti_TNF": [
        'Infliximab_medcount', 'Adalimumab_medcount',
        'Golimumab_medcount', 'Certolizumabpegol_medcount'
    ],
    "Anti_Interleukin": [
        'Ustekinumab_medcount', 'Mirikizumab_medcount', 'Risankizumab_medcount'
    ],
    "Anti_Integrin": [
        'Vedolizumab_medcount', 'Natalizumab_medcount'
    ],
    "JAKi": [
        'Upadacitinib_medcount', 'Tofacitinib_medcount'
    ],
    "S1PRM": [
        'Ozanimod_medcount', 'Etrasimod_medcount'
    ],
    "Steroid": [
        'Budesonide_medcount'
    ]
}

for group, cols in medication_groups.items():
    df[f'{group}_prescriptioncount'] = df[cols].sum(axis=1)
    df[group] = (df[f'{group}_prescriptioncount'] > 0).astype(int)


meds_datesweightgroups = df.copy()
meds_datesweightgroups

### Labs and BMI Categorization

The maximum Lab values and median BMI generated in the preceding preprocessing steps were converted into categorical variables using specified thresholds.

Variables created:

- `maxCRP_Classification`
- `maxESR_Classification`
- `maxFecal calprotectin_Classification`
- `maxFecal lactoferrin_classification`
- `bmi_Classification`

Missing measurements were assigned to a separate `Missing` category.

The resulting dataframe, labsclassification, combined the previously derived patient level features with the categorized labs and BMI variables.

In [ ]:
df = meds_datesweightgroups.copy()

numeric_columns = [
    "CRP",
    "ESR",
    "Fecal calprotectin",
    "Fecal lactoferrin",
    "median_bmi"
]

df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric,errors="coerce")

# Classify maximum CRP values
def classify_CRP(value):
    if pd.isna(value):
        return 'Missing'
    elif 0 <= value < 5:
        return 'Normal'
    elif 5 <= value < 10:
        return 'High'
    elif 10 <= value < 15:
        return 'Very high'
    elif value >= 15:
        return 'Extremely high'
    else:
        return 'unknown'


def classify_ESR(value):
    if pd.isna(value):
        return 'Missing'
    elif 0 <= value < 20:
        return 'Normal'
    elif 20 <= value < 30:
        return 'High'
    elif 30 <= value < 40:
        return 'Very high'
    elif value >= 40:
        return 'Extremely high'
    else:
        return 'unknown'


def classify_fecalcal(value):
    if pd.isna(value):
        return 'Missing'
    elif 0 <= value < 50:
        return 'Normal'
    elif 50 <= value < 250:
        return 'Abnormal'
    elif 250 <= value < 800:
        return 'High'
    elif value >= 800:
        return 'Extremely high'
    else:
        return 'unknown'


def classify_fecallacto(value):
    if pd.isna(value):
        return 'Missing'
    elif value <= 0:
        return 'Not Detected'
    elif value > 0:
        return 'Detected'
    else:
        return 'unknown'


def classify_BMI(value):
    if pd.isna(value):
        return 'Missing'
    elif value < 18.5:
        return 'Underweight'
    elif 18.5 <= value < 30:
        return 'Normal'
    elif value >= 30:
        return 'Severely obese'
    else:
        return 'unknown'

df['maxCRP_Classification'] = df['CRP'].apply(classify_CRP)
df['maxESR_Classification'] = df['ESR'].apply(classify_ESR)
df['maxFecal calprotectin_Classification'] = df['Fecal calprotectin'].apply(classify_fecalcal)
df['maxFecal lactoferrin_classification'] = df['Fecal lactoferrin'].apply(classify_fecallacto)
df['bmi_Classification'] = df['median_bmi'].apply(classify_BMI)

df = df.drop(columns=['median_bmi', 'CRP', 'ESR', 'Fecal calprotectin', 'Fecal lactoferrin'])

labsclassification = df.copy()
labsclassification

### Age at First IBD Diagnosis and Age Categorization

The first IBD diagnosis date was defined as the earliest available date between the diagnosis start date and the user entered diagnosis date which were extracted in the preceding steps.

Age at first IBD diagnosis was calculated in completed years between `BirthDate` and the derived first IBD diagnosis date.
AgeInYears and age at first IBD diagnosis were then grouped using the prespecified age categories.

In [ ]:

df = labsclassification.copy()

date_columns = ["BirthDate", "diagnosisStartDate", "diagnosisEnterDate"]

df[date_columns] = df[date_columns].apply(pd.to_datetime)

# earliest available IBD diagnosis date
df["date.IBDfirstDiagnosis"] = df[["diagnosisStartDate","diagnosisEnterDate"]].min(axis=1)
df["AgeAtIBDFirstDiagnosis"] = (df["date.IBDfirstDiagnosis"].dt.year - df["BirthDate"].dt.year)

birthday_not_reached = ((df["date.IBDfirstDiagnosis"].dt.month< df["BirthDate"].dt.month) |
                        ((df["date.IBDfirstDiagnosis"].dt.month == df["BirthDate"].dt.month) &
                         (df["date.IBDfirstDiagnosis"].dt.day< df["BirthDate"].dt.day)))

df["AgeAtIBDFirstDiagnosis"] = (df["AgeAtIBDFirstDiagnosis"]- birthday_not_reached.astype(int))


assert not (
    df["AgeAtIBDFirstDiagnosis"].dropna() < 0
).any(), "Diagnosis date precedes birth date for one or more patients."


df['AgeInYears'] = pd.to_numeric(df['AgeInYears'], errors='coerce')
df['AgeAtIBDFirstDiagnosis'] = pd.to_numeric(df['AgeAtIBDFirstDiagnosis'], errors='coerce')


diagnosis_bins = [-float('inf'), 6, 18, 50, float('inf')]
diagnosis_labels = [
    'Less than or equal to 6',
    'Greater than 6 and less than or equal to 18',
    'Greater than 18 and less than or equal to 50',
    'Greater than 50'
]

# Current age bins
age_bins = [-float('inf'), 18, 35, 50, 65, float('inf')]
age_labels = [
    'Less than 18',
    'Greater than or equal to 18 and less than 35',
    'Greater than or equal to 35 and less than 50',
    'Greater than or equal to 50 and less than 65',
    'Greater than or equal to 65'
]

df['Current_Age'] = pd.cut(df['AgeInYears'], bins=age_bins, labels=age_labels, right=False)
df['ageatIBDDiagnosis'] = pd.cut(df['AgeAtIBDFirstDiagnosis'], bins=diagnosis_bins, labels=diagnosis_labels)

df['Current_Age'] = df['Current_Age'].cat.add_categories('Missing').fillna('Missing')
df['ageatIBDDiagnosis'] = df['ageatIBDDiagnosis'].cat.add_categories('Missing').fillna('Missing')

df = df.drop(columns=['AgeInYears', 'AgeAtIBDFirstDiagnosis'])

ageclassification_df = df.copy()
ageclassification_df

### Ethnicity and Race Categorization

Ethnicity and first race were combined into a single categorical variable, `Ethnicity_Race`.

The following prespecified combinations were used:

- Hispanic or Latino with Asian or Caucasian race was categorized as `Hispanic or Latino`
- Not Hispanic or Latino with Asian race
- Not Hispanic or Latino with Black race
- Not Hispanic or Latino with Caucasian race

All other ethnicity-race combinations were categorized as `Others`.

In [ ]:

# Continue from the age-classification step
df = ageclassification_df.copy()

keep_combinations = [
    ('Hispanic or Latino', 'Asian'),
    ('Hispanic or Latino', 'Caucasian'),
    ('Not Hispanic or Latino', 'Asian'),
    ('Not Hispanic or Latino', 'Black'),
    ('Not Hispanic or Latino', 'Caucasian'),
]


keep_set = set(keep_combinations)
pairs = pd.Series(
    list(zip(df["Ethnicity"], df["FirstRace"])),
    index=df.index
)

df['Ethnicity_Race'] = 'Others'

mask = pairs.isin(keep_set)

df.loc[mask & (df['Ethnicity'] == 'Hispanic or Latino'), 'Ethnicity_Race'] = 'Hispanic or Latino'

df.loc[mask & (df['Ethnicity'] == 'Not Hispanic or Latino'), 'Ethnicity_Race'] = (
        df['Ethnicity'] + ' - ' + df['FirstRace'])

ethnicityrace_classification = df.copy()
ethnicityrace_classification

### Final Missing Value Handling

After the patient level tables were joined, missing values in count and binary variables were set to `0` when no record was found for the patient.

Missing lab values, BMI, dates, and categorical variables were not replaced with zero.

In [ ]:

df = ethnicityrace_classification.copy()

zero_cols = ['CD_i', 'UC_i', 'CDUCWeight_i', 'CD_e', 'UC_e', 'CDUCWeight_e', 'externaldiagnosis_count',
             'timesSeenbyIBDProv', 'NumHMEncounters', 'seenbyDietitian', 'IBD_OP', 'outpatientvisits_SeenbyIBDProv',
             'futureoutpatientvisits_SeenbyIBDProv', 'newpatientFuturevisits_SeenbyIBDProv', 'SeenbyGastSM12provider',
             'Adalimumab_medcount', 'Azathioprine_medcount', 'Balsalazide_medcount', 'Certolizumabpegol_medcount',
             'Golimumab_medcount', 'Infliximab_medcount', 'Mercaptopurine_medcount', 'Mesalamine_medcount',
             'Methotrexate_medcount', 'Mirikizumab_medcount', 'Natalizumab_medcount', 'Olsalazine_medcount',
             'Ozanimod_medcount', 'Risankizumab_medcount', 'Sulfasalazine_medcount', 'Tofacitinib_medcount',
             'Upadacitinib_medcount', 'Ustekinumab_medcount', 'Vedolizumab_medcount', 'Etrasimod_medcount',
             'Budesonide_medcount', 'Hydrocortisone_medcount', 'Methylprednisolone_medcount', 'Prednisone_medcount',
             'Peripheralarthropathies_seencount', 'Axialarthropathies_seencount', 'Erythemanodosum_seencount',
             'Pyodermagngrenosum_seencount', 'Sweetssyndrome_seencount', 'Apthousstomatitis_seencount',
             'Primarysclerosischolangitis_seencount', 'Episcleritis_seencount', 'AnteriorUveitis_seencount',
             'Iritis_seencount', 'Scleritis_seencount', 'Psoriasis_seencount', 'psoriaticarthritis_seencount',
             'PressureUlcer_seencount', 'NPchroniculcerlowerlimb_seencount', 'NPchroniculcerskin_seencount',
             'VascularDisordersIntestine_seencount', 'DiverticularDiseaseIntestine_seencount', 'IBS_seencount',
             'bacterialintestinalinfections_seencount', 'EnterocolitisClostridiumDifficile_seencount',
             'SpecifiedBacterialInfections_seencount', 'BacterialintestinalInfectionUnspecified_seencount',
             'Infectiousgastroenteritiscolitis_seencount', 'coloncancer_seencount', 'historyofColoncancer_seencount',
             'Numberofendoscopyprocedures', 'endoscopyrecords_count', 'Endoscopy_Aphthous ulcer',
             'Endoscopy_Anastomosis', 'Endoscopy_Pouchitis', 'Endoscopy_ulcerated', 'Endoscopy_erythema',
             'Endoscopy_Loss of vascularity', 'Endoscopy_Friable mucosa', 'Endoscopy_mayo',
             'Endoscopy_contact bleeding', 'Endoscopy_rutgeerts', 'Endoscopy_Ileo-Colonic', 'Endoscopy_SES-CD',
             'colon&rectal_seencount', 'Gastroenterology_recordscount', 'Gastroenterology_Ileitis',
             'Gastroenterology_Abscess', 'Gastroenterology_Fistula', 'Gastroenterology_Stricture',
             'Gastroenterology_Proctocolectomy', 'Gastroenterology_Colectomy', 'Gastroenterology_Ileostomy',
             'Gastroenterology_Colitis', 'Gastroenterology_Wall Thickening', 'Gastroenterology_Ischemia',
             'Gastroenterology_Ischemic', 'Gastroenterology_Ischaemia', 'Gastroenterology_Ischaemic',
             'Gastroenterology_Ileocecal resection', 'Gastroenterology_Hemicolectomy',
             'Gastroenterology_Diverting ileostomy', 'Gastroenterology_Proctosigmoiditis',
             'Gastroenterology_Total colectomy', 'Gastroenterology_Total proctocolectomy',
             'Gastroenterology_Ileorectal', 'Gastroenterology_Fibrostenotic', 'Gastroenterology_J-pouch',
             'Gastroenterology_Pouchitis', 'Gastroenterology_Ileocolonic Anastomosis',
             'Gastroenterology_Ileal Pouch Anal Anastomosis', 'Gastroenterology_Ileoanal', 'Imaging_recordscount',
             'Imaging_Ileitis', 'Imaging_Abscess', 'Imaging_Fistula', 'Imaging_Stricture', 'Imaging_Proctocolectomy',
             'Imaging_Colectomy', 'Imaging_Ileostomy', 'Imaging_Colitis', 'Imaging_Wall Thickening', 'Imaging_Ischemia',
             'Imaging_Ischemic', 'Imaging_Ischaemia', 'Imaging_Ischaemic', 'Imaging_Ileocecal resection',
             'Imaging_Hemicolectomy', 'Imaging_Diverting ileostomy', 'Imaging_Proctosigmoiditis',
             'Imaging_Total colectomy', 'Imaging_Total proctocolectomy', 'Imaging_Ileorectal', 'Imaging_Fibrostenotic',
             'Imaging_J-pouch', 'Imaging_Pouchitis', 'Imaging_Ileocolonic Anastomosis',
             'Imaging_Ileal Pouch Anal Anastomosis', 'Imaging_Ileoanal', 'pathology_recordscount', 'pathology_Chronic',
             'pathology_Ileitis', 'pathology_Pouchitis', 'pathology_Colitis', 'pathology_Granuloma',
             'pathology_Granulomatous', 'pathology_Crypt distortion', 'pathology_Crypt Abscess']

df[zero_cols] = df[zero_cols].fillna(0)
df = df.drop(columns=['PrimaryMrn', 'Name','BirthDate','DeathDate','Ethnicity','FirstRace','PostalCode','City','StateOrProvince','diagnosisStartDate','diagnosisEnterDate', 'date.IBDfirstDiagnosis'])


final_analytic_df = df.copy()

final_analytic_df

assert final_analytic_df["PatientDurableKey"].notna().all()
assert not final_analytic_df["PatientDurableKey"].duplicated().any()

print(f"Final patients: {len(final_analytic_df):,}")
print(f"Final variables: {final_analytic_df.shape[1]:,}")

final_analytic_df

Generative AI was used for limited grammar and language editing.

The same SQL data extraction criteria, data cleaning procedures and preprocessing steps workflow were applied consistently to both the pre 2022 and post 2022 patient cohorts.

The pre 2022 cohort included patients assigned to the period ending December 31, 2021, whereas the post 2022 cohort included patients assigned to the period beginning January 1, 2022.